# Real-Time Sign Language Recognition - Deep Learning Pipeline

This Jupyter Notebook guides you through the process of:
1. Loading the normalized hand landmark dataset.
2. Training a Multi-Layer Perceptron (MLP) model in PyTorch.
3. Evaluating its performance (Classification metrics & Confusion Matrix).
4. Explaining exporting options to run the model in diverse environments.

---

### Step 1: Import Dependencies
Let's import PyTorch, NumPy, Pandas, Scikit-learn, and Matplotlib.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import itertools

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

### Step 2: Define the Network Architecture
We will define a 4-layer Fully Connected Neural Network (MLP) with Batch Normalization and Dropout layers to prevent overfitting.

In [ ]:
class SignLanguageMLP(nn.Module):
    def __init__(self, num_classes):
        super(SignLanguageMLP, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(63, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            
            nn.Linear(32, num_classes)
        )
        
    def forward(self, x):
        return self.fc(x)

### Step 3: Dataset Loading & Preparation
We define the PyTorch dataset helper. If the `sign_landmarks.csv` doesn't exist, we will automatically generate mockup data for testing purposes.

In [ ]:
class LandmarkDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

csv_path = 'data/sign_landmarks.csv'

# Auto-generate mockup data if dataset is empty
if not os.path.exists(csv_path):
    print("Dataset CSV not found. Generating a mockup dataset for verification...")
    classes = ['A', 'B', 'C', 'L', 'Y']
    data = []
    for cls in classes:
        for _ in range(120):
            vec = np.random.normal(0.2, 0.1, 63)
            data.append([cls] + list(vec))
    os.makedirs('data', exist_ok=True)
    df = pd.DataFrame(data, columns=['label'] + [f'coord_{i}' for i in range(63)])
    df.to_csv(csv_path, index=False)

# Load data
df = pd.read_csv(csv_path)
X = df.iloc[:, 1:].values
y_raw = df.iloc[:, 0].values

# Encode Labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
classes = label_encoder.classes_.tolist()
print("Dataset Loaded! Classes:", classes)

### Step 4: Model Training
Let's split our dataset, create data loaders, define the loss function and optimizer, and run the training cycle.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

train_loader = DataLoader(LandmarkDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(LandmarkDataset(X_val, y_val), batch_size=32, shuffle=False)

model = SignLanguageMLP(num_classes=len(classes))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-4)

epochs = 30
train_losses, val_losses, val_accs = [], [], []

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(inputs), targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
    
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            outputs = model(inputs)
            val_loss += criterion(outputs, targets).item() * inputs.size(0)
            correct += (outputs.argmax(dim=1) == targets).sum().item()
            total += targets.size(0)
            
    train_losses.append(train_loss / len(train_loader.dataset))
    val_losses.append(val_loss / len(val_loader.dataset))
    val_accs.append(correct / total)
    
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} | Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f} | Accuracy: {val_accs[-1]*100:.1f}%")

### Step 5: Visualizing Training Curves
Let's plot loss and validation accuracy curves.

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss', color='indigo')
plt.plot(val_losses, label='Val Loss', color='cyan')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(val_accs, label='Accuracy', color='green')
plt.title('Validation Accuracy')
plt.legend()
plt.show()

### Step 6: Evaluation Metrics
Let's print the classification report and plot a Confusion Matrix.

In [ ]:
model.eval()
inputs_val = torch.tensor(X_val, dtype=torch.float32)
with torch.no_grad():
    preds = model(inputs_val).argmax(dim=1).numpy()

print("\nClassification Report:\n")
print(classification_report(y_val, preds, target_names=classes))

cm = confusion_matrix(y_val, preds)
plt.figure(figsize=(6, 6))
plt.imshow(cm, cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks(range(len(classes)), classes, rotation=45)
plt.yticks(range(len(classes)), classes)
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()

### Step 7: Model Export Options

To use this PyTorch model in the browser or mobile devices, you can convert it to alternative formats:

#### 1. ONNX (Open Neural Network Exchange)
ONNX allows running model inference in C++, Javascript, Python, and other backends efficiently.
```python
dummy_input = torch.randn(1, 63)
torch.onnx.export(model, dummy_input, "data/model.onnx", input_names=['landmarks'], output_names=['predictions'])
```

#### 2. TensorFlow.js (for Web browser)
You can convert the PyTorch model to Keras and export to TF.js format using standard converter libraries.